In [198]:
import pandas as pd
df_transactions = pd.read_parquet('ledger_transactions.parquet')

df_transactions.info()



<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   transaction_id     5000 non-null   str           
 1   applicant_id       5000 non-null   str           
 2   timestamp          5000 non-null   datetime64[us]
 3   amount             5000 non-null   float64       
 4   terminal_location  5000 non-null   str           
 5   category           5000 non-null   str           
dtypes: datetime64[us](1), float64(1), str(4)
memory usage: 401.9 KB


In [ ]:
import pandas as pd
df_transactions = pd.read_parquet('ledger_transactions.parquet')

mean_amount = df_transactions.groupby("category")["amount"].transform("mean")
std_amount = df_transactions.groupby("category")["amount"].transform("std")

upper_limit = mean_amount + (3 * std_amount)
is_outlier = df_transactions["amount"] > upper_limit
df_transactions = df_transactions[~is_outlier]

df_transactions.info()





,transaction_id,applicant_id,timestamp,amount,terminal_location,category
0,TX00000,APP004,2026-08-01 00:00:00,73.998193,Tbilisi Central,Groceries
1,TX00001,APP005,2026-08-01 00:01:00,117.054648,Kutaisi Station,Groceries
2,TX00002,APP003,2026-08-01 00:02:00,516.611718,Tbilisi Central,Retail
3,TX00003,APP005,2026-08-01 00:03:00,10.332549,Online Gateway,Travel
4,TX00004,APP005,2026-08-01 00:04:00,156.754191,Online Gateway,Retail
5,TX00005,APP002,2026-08-01 00:05:00,11.549235,Online Gateway,Utilities
6,TX00006,APP003,2026-08-01 00:06:00,70.613129,Tbilisi Central,Retail
7,TX00007,APP003,2026-08-01 00:07:00,244.304230,Online Gateway,Retail
8,TX00008,APP003,2026-08-01 00:08:00,85.234378,Online Gateway,Groceries
9,TX00009,APP005,2026-08-01 00:09:00,880.778019,Kutaisi Station,Retail


In [150]:
import pandas as pd
import json

# read data
df_applications = pd.read_csv('applications.csv')
with open('bureau_scores.json') as f:
    raw_bureau_scores = json.load(f)

# normalize collumn names and clean dupplicates
df_bureau_scores = pd.json_normalize(raw_bureau_scores)
df_applications.columns = df_applications.columns.str.replace(' ', '')

transformed_income = df_applications.groupby("education_level")["monthly_income"].transform('median')
filled_df_applications = df_applications.fillna({'monthly_income': transformed_income})

filled_df_applications["submission_date"]= pd.to_datetime(filled_df_applications['submission_date'], format='mixed',dayfirst=True, errors='coerce')
sorted_applications = filled_df_applications.sort_values(by='submission_date', ascending=False)
drop_duplicates_applications = sorted_applications.drop_duplicates(subset='applicant_id', keep='first')


# join two tables
clean_applications_with_scores = df_bureau_scores.set_index("applicant_id").join(drop_duplicates_applications.set_index("applicant_id"),how='inner')


clean_applications_with_scores.head(20)



,credit_profile.bureau_score,credit_profile.past_defaults,credit_profile.hard_inquiries,full_name,submission_date,monthly_income,education_level,employment_status
applicant_id,,,,,,,,
APP001,720,0,1,Giorgi Beridze,2026-03-09,2500.0,Bachelor,Employed
APP002,580,2,4,Nino Kalandadze,2026-09-29,1800.0,Master,Employed
APP003,640,1,2,Levan Jishkariani,2026-04-09,1200.0,High School,Unemployed
APP004,790,0,0,Mariam Lomidze,2026-05-09,4200.0,PhD,Employed
APP005,550,3,5,Irakli Mgeladze,2026-09-26,2500.0,Bachelor,Self-Employed
APP006,710,0,1,Salome Kankia,2026-11-16,3100.0,Master,Employed
